## **Bronze Layer**

In [13]:
import fmlv

StatementMeta(, 8eaaf8ac-35b6-4901-94a2-4156975030b0, 15, Finished, Available, Finished, False)

In [14]:
%%sql
ALTER TABLE customers SET TBLPROPERTIES ('delta.enablechangedatafeed' = 'true');
ALTER TABLE order_items SET TBLPROPERTIES ('delta.enablechangedatafeed' = 'true');
ALTER TABLE orders SET TBLPROPERTIES ('delta.enablechangedatafeed' = 'true');
ALTER TABLE products SET TBLPROPERTIES ('delta.enablechangedatafeed' = 'true');

StatementMeta(, 8eaaf8ac-35b6-4901-94a2-4156975030b0, 19, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [3]:
%%sql
CREATE SCHEMA mlv.Bronze

StatementMeta(, fda95dfc-4c09-4c3e-b4fc-3cbbdddd2065, 7, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [12]:
@fmlv.materialized_lake_view(
    name = "mlv.Bronze.orders",
    table_properties={"delta.enablechangedatafeed":"true"}
)

def orders():
    df = spark.read.table("mlv.dbo.orders")
    return df

StatementMeta(, 7f921d31-8dfe-442f-b8af-9915c873f775, 17, Finished, Available, Finished, False)

AnalysisException: [MLV_ALREADY_EXISTS] A materialized lake view with this name already exists. Use a different name or remove the existing view before creating a new one. failureType: UserError errorDetails: No additional trace available

In [13]:
@fmlv.materialized_lake_view(
    name = "mlv.Bronze.products",
    table_properties={"delta.enablechangedatafeed":"true"}
)

def products():
    df = spark.read.table("mlv.dbo.products")
    return df

StatementMeta(, 7f921d31-8dfe-442f-b8af-9915c873f775, 18, Finished, Available, Finished, False)

AnalysisException: [MLV_ALREADY_EXISTS] A materialized lake view with this name already exists. Use a different name or remove the existing view before creating a new one. failureType: UserError errorDetails: No additional trace available

In [8]:
@fmlv.materialized_lake_view(
    name = "mlv.Bronze.order_items",
    table_properties={"delta.enablechangedatafeed":"true"}
)

def order_items():
    df = spark.read.table("mlv.dbo.order_items")
    return df

StatementMeta(, fda95dfc-4c09-4c3e-b4fc-3cbbdddd2065, 13, Finished, Available, Finished, False)

## **Silver Layer**

In [12]:
%%sql
CREATE SCHEMA mlv.Silver

StatementMeta(, fda95dfc-4c09-4c3e-b4fc-3cbbdddd2065, 17, Finished, Available, Finished, False)

Error: [SCHEMA_ALREADY_EXISTS] Cannot create schema `chimcobldhq2ah2g40rj0c10e1p62orkd5hma9bddhr2akr9dhr6asg` because it already exists.
Choose a different name, drop the existing schema, or add the IF NOT EXISTS clause to tolerate pre-existing schema.

In [20]:
@fmlv.materialized_lake_view(
    name = "mlv.Silver.orders_obt",
    table_properties = {"delta.enalblechangedatafeed":"true"}
)


def order_obt():


    #Bronze orders
    df_orders = spark.read.table("mlv.bronze.orders")
    df_orders = df_orders.select("order_id","customer_id","order_status")

    #Bronze order items
    df_oi = spark.read.table("mlv.bronze.order_items")
    df_oi = df_oi.select("order_id","order_item_id","product_id","price")

    #Bronze product
    df_pro = spark.read.table("mlv.bronze.products")
    df_pro = df_pro.select("product_id","product_category_name")


    #JOINS
    df_join = df_orders.join(df_oi,df_orders['order_id']==df_oi['order_id'],"left")\
                       .join(df_pro,df_oi['product_id']==df_pro['product_id'])\
                       .select("orders.order_id","orders.customer_id","orders.order_status","order_items.order_item_id","order_items.price","products.product_category_name")
    
    return df_join
    


    

StatementMeta(, 7f921d31-8dfe-442f-b8af-9915c873f775, 25, Finished, Available, Finished, False)

In [5]:
#Bronze orders
df_orders = spark.read.table("mlv.bronze.orders")
df_orders = df_orders.select("order_id","customer_id","order_status")

    #Bronze order items
df_oi = spark.read.table("mlv.bronze.order_items")
df_oi = df_oi.select("order_id","order_item_id","product_id","price")

    #Bronze product
df_pro = spark.read.table("mlv.bronze.products")
df_pro = df_pro.select("product_id","product_category_name")


    #JOINS
df_join = df_orders.join(df_oi,df_orders['order_id'] == df_oi['order_id'],"left")\
                   .join(df_pro,df_oi['product_id'] == df_pro['product_id'])\
                   .select("orders.order_id","orders.customer_id","orders.order_status","order_items.price","products.product_category_name")
    
display(df_join)

StatementMeta(, 7f921d31-8dfe-442f-b8af-9915c873f775, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b862e636-31f3-42e6-bb52-ffcc1fbb21dd)

## **GOLD LAYER**

In [1]:
from pyspark.sql.functions import *

StatementMeta(, 8eaaf8ac-35b6-4901-94a2-4156975030b0, 3, Finished, Available, Finished, False)

In [17]:
%%sql
create SCHEMA gold

StatementMeta(, 8eaaf8ac-35b6-4901-94a2-4156975030b0, 22, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [18]:
@fmlv.materialized_lake_view(
    name = "mlv.gold.top_categories",
    table_properties = {"delta.enalblechangedatafeed":"true"}
)
@fmlv.check("price","total_sales > 0","DROP")
def top_categories():

   df = spark.read.table("mlv.Silver.orders_obt")
   df = df.groupBy("product_category_name").agg(sum(col("price")).alias("total_sales")).sort("total_sales",ascending = False)

   return df

StatementMeta(, 8eaaf8ac-35b6-4901-94a2-4156975030b0, 23, Finished, Available, Finished, False)

In [19]:
%%sql
CREATE MATERIALIZED LAKE VIEW if NOT EXISTS mlv.gold.delivery_status
(CONSTRAINT status_check CHECK(order_status IS NOT NULL)on MISMATCH DROP)
AS
SELECT
   order_status,
   count(order_id) as total_sales
FROM
   mlv.silver.orders_obt
GROUP BY
    order_status

StatementMeta(, 8eaaf8ac-35b6-4901-94a2-4156975030b0, 24, Finished, Available, Finished, False)

<Spark SQL result set with 11 rows and 2 fields>